> **Public release note.** Notebook outputs have been removed because the underlying Malaysian Motor claims data are confidential. Execution counts have also been cleared to provide clean public versions of the notebooks. Local user-specific paths and individual claim identifiers have been removed. The repository documents the data-processing, modelling and validation workflow used in the dissertation, but the numerical results cannot be reproduced end-to-end without the confidential input data.


# 05.4A — Clean Snapshot Model Inputs

**Purpose:** rebuild the fixed-maturity modelling inputs from the upstream
`model_ready_clean_features_flagged` files using an explicit feature whitelist.

This notebook prevents future-information leakage by saving predictors,
targets and metadata in separate files. It replaces the previous practice of
saving complete train/test dataframes and then rediscovering features later.

Main outputs for each snapshot:

- `X_train.parquet` and `X_test.parquet`: approved predictors only;
- `targets_train.parquet` and `targets_test.parquet`: outcomes only;
- `meta_train.parquet` and `meta_test.parquet`: identifiers, timing and split fields;
- `feature_manifest.json`: the locked feature definition;
- `clean_snapshot_input_audit.csv`: data and leakage checks.

The first cell sets up the paths, output folders, snapshot maturities and temporal train-test definitions before the claim-level data are loaded.

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

PROJECT_FOLDER = Path(
    "/path/to/BI_large_claims_project"
)

MODEL_READY_FOLDER = PROJECT_FOLDER / "processed" / "model_ready"

OUTPUT_FOLDER = (
    PROJECT_FOLDER
    / "processed"
    / "chapter5_outputs"
    / "section_5_4A_clean_snapshot_model_inputs"
)
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

SNAPSHOTS = ["DEV_QTR_4", "DEV_QTR_8", "DEV_QTR_12"]

TRAIN_LABEL = "train_up_to_2019_Q4"
TEST_LABEL = "test_2020_Q1_to_2022_Q4"

print("Model-ready folder exists:", MODEL_READY_FOLDER.exists())
print("Output folder:", OUTPUT_FOLDER)

## Locked feature definition

The feature list is a **whitelist**, not a list inferred from all available
columns. Every predictor must be known at the valuation snapshot.

`CLASS`, `cover`, `NATLOSS` and `ACC_QTR` are treated as categorical.
`DEV_QTR` is excluded because a separate model is fitted at each fixed
snapshot, making it constant within each model.

Log-transformed copies are not included. For tree models they add no new
ordering information and can obscure interpretation.

This section explains which claim characteristics the models can use. It separates categorical variables from numeric ones and lists the prediction targets and supporting metadata. To prevent data leakage, it checks that future information, claim identifiers, and valuation fields are not included as model features. If it finds a forbidden or duplicate variable, the notebook stops right away.

In [ ]:
CATEGORICAL_FEATURES = [
    "CLASS",
    "cover",
    "NATLOSS",
    "ACC_QTR",
]

NUMERIC_FEATURES = [
    "ACC_YEAR",
    "PAIDLS",
    "BALOS",
    "TAG_INC_LARGE",
    "TAG_PAIDLS_LARGE",
    "CLAIMS_CNT",
    "SETTLED_CNT",
    "CUM_PAIDLS",
    "CUM_INC_AMT",
    "CUM_INC_NON_LARGE",
    "CUM_INC_LARGE",
    "CLAIMS_CNT_LARGE",
    "SETTLED_CNT_LARGE",
    "CUM_PAIDLS_NON_LARGE",
    "CUM_PAIDLS_LARGE",
    "is_bi_excess_at_snapshot",
]

FEATURE_COLUMNS = CATEGORICAL_FEATURES + NUMERIC_FEATURES

TARGET_LATEST = "target_latest_CUM_INC_LARGE"
TARGET_FUTURE_CLEAN = "future_excess_incurred_development_clean"
TARGET_CLASS_CLEAN = "is_bi_excess_at_latest_clean"
SNAPSHOT_EXCESS = "snapshot_CUM_INC_LARGE"

META_CANDIDATES = [
    "SOURCE_FILE",
    "CLAIMS_KEY",
    "ACC_YEAR",
    "ACC_QTR",
    "DEV_QTR",
    "ACCIDENT_QTR_INDEX",
    "VALUATION_QTR_INDEX",
    "VALUATION_YEAR",
    "VALUATION_QTR",
    "VALUATION_YEAR_QTR",
    "valuation_split",
]

FORBIDDEN_FEATURES = {
    "SOURCE_FILE",
    "CLAIMS_KEY",
    "target_latest_CUM_INC_LARGE",
    "future_excess_incurred_development",
    "future_excess_incurred_development_clean",
    "is_bi_excess_at_latest",
    "is_bi_excess_at_latest_clean",
    "is_extreme_case_reserve_movement",
    "ACCIDENT_QTR_INDEX",
    "VALUATION_QTR_INDEX",
    "VALUATION_YEAR",
    "VALUATION_QTR",
    "VALUATION_YEAR_QTR",
    "valuation_split",
}

forbidden_used = sorted(set(FEATURE_COLUMNS).intersection(FORBIDDEN_FEATURES))
if forbidden_used:
    raise ValueError(f"Forbidden fields entered the feature whitelist: {forbidden_used}")

if len(FEATURE_COLUMNS) != len(set(FEATURE_COLUMNS)):
    raise ValueError("Duplicate feature names found.")

print("Approved feature count:", len(FEATURE_COLUMNS))
print("Categorical:", CATEGORICAL_FEATURES)
print("Numeric:", NUMERIC_FEATURES)

This section creates the file paths for the three development snapshots, checks that all needed input files are there, and loads each Parquet file into memory. It also prints the size and accident-year range of each dataset to quickly confirm the files look correct before starting the modeling.

In [ ]:
model_ready_paths = {
    snapshot: MODEL_READY_FOLDER / f"{snapshot}_model_ready_clean_features_flagged.parquet"
    for snapshot in SNAPSHOTS
}

raw_data = {}

for snapshot, path in model_ready_paths.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing input for {snapshot}: {path}")

    df = pd.read_parquet(path)
    raw_data[snapshot] = df

    print(
        snapshot,
        "shape=", df.shape,
        "AY=", (df["ACC_YEAR"].min(), df["ACC_YEAR"].max())
    )

## Clean targets and structural zero values

Missing BI Excess amounts represent valid non-excess claims and are converted
to zero. The future-development target is recreated directly as:

`latest observed incurred BI Excess − snapshot incurred BI Excess`.

Any pre-existing future-development field is checked against this clean
calculation but is never used as a predictor.

This step performs the main data cleaning and checking for each snapshotp, it recalculates the BI Excess targets from the financial data, makes the data types consistent, checks that the original targets match the recalculated values, and makes sure there is no overlap between the training and test claims.

For each snapshot, it creates an audit summary. This summary includes the number of rows, the accident-year coverage, duplicate claims, BI Excess totals, future development totals, and any target mismatches. This allows the modelling datasets to be checked before use.

In [ ]:
cleaned_data = {}
audit_rows = []

for snapshot, original_df in raw_data.items():
    df = original_df.copy()
    df["MODEL_ROW_ID"] = np.arange(len(df), dtype=np.int64)

    required = set(FEATURE_COLUMNS + [
        "SOURCE_FILE",
        "CLAIMS_KEY",
        "DEV_QTR",
        "valuation_split",
        TARGET_LATEST,
    ])
    missing_required = sorted(required.difference(df.columns))
    if missing_required:
        raise ValueError(f"{snapshot}: missing required columns {missing_required}")

    # Structural zeros in the BI Excess layer.
    for col in ["CUM_INC_LARGE", "CUM_PAIDLS_LARGE", TARGET_LATEST]:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

    # Recreate snapshot and targets from the financial fields.
    df[SNAPSHOT_EXCESS] = df["CUM_INC_LARGE"].astype(float)
    df[TARGET_FUTURE_CLEAN] = (
        df[TARGET_LATEST].astype(float) - df[SNAPSHOT_EXCESS]
    )
    df[TARGET_CLASS_CLEAN] = (df[TARGET_LATEST] > 0).astype("int8")
    df["is_bi_excess_at_snapshot"] = (
        df[SNAPSHOT_EXCESS] > 0
    ).astype("int8")

    # Compare with upstream target/flag where available.
    target_check_max_abs_diff = np.nan
    target_check_mismatch_count = np.nan
    if "future_excess_incurred_development" in df.columns:
        upstream_future = pd.to_numeric(
            df["future_excess_incurred_development"], errors="coerce"
        ).fillna(0.0)
        target_diff = upstream_future - df[TARGET_FUTURE_CLEAN]
        target_check_max_abs_diff = float(target_diff.abs().max())
        target_check_mismatch_count = int((target_diff.abs() > 0.01).sum())

        if target_check_mismatch_count > 0:
            raise ValueError(
                f"{snapshot}: upstream future target differs from clean "
                f"calculation in {target_check_mismatch_count:,} rows; "
                f"maximum absolute difference={target_check_max_abs_diff:,.2f}"
            )

    class_flag_mismatch_count = np.nan
    if "is_bi_excess_at_latest" in df.columns:
        upstream_flag = (
            pd.to_numeric(df["is_bi_excess_at_latest"], errors="coerce")
            .fillna(0)
            .astype(int)
        )
        class_flag_mismatch_count = int(
            (upstream_flag != df[TARGET_CLASS_CLEAN]).sum()
        )

    # Explicit data types.
    for col in CATEGORICAL_FEATURES:
        df[col] = df[col].astype("string")

    for col in NUMERIC_FEATURES:
        if col == "is_bi_excess_at_snapshot":
            df[col] = df[col].fillna(0).astype("int8")
        else:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    duplicate_claims = int(
        df.duplicated(subset=["SOURCE_FILE", "CLAIMS_KEY"], keep=False).sum()
    )

    split_counts = df["valuation_split"].value_counts(dropna=False)
    unexpected_splits = sorted(
        set(split_counts.index.astype(str)).difference({TRAIN_LABEL, TEST_LABEL})
    )
    if unexpected_splits:
        raise ValueError(
            f"{snapshot}: unexpected valuation_split values {unexpected_splits}"
        )

    train_mask = df["valuation_split"].eq(TRAIN_LABEL)
    test_mask = df["valuation_split"].eq(TEST_LABEL)

    train_keys = set(
        zip(df.loc[train_mask, "SOURCE_FILE"], df.loc[train_mask, "CLAIMS_KEY"])
    )
    test_keys = set(
        zip(df.loc[test_mask, "SOURCE_FILE"], df.loc[test_mask, "CLAIMS_KEY"])
    )
    overlap_count = len(train_keys.intersection(test_keys))
    if overlap_count:
        raise ValueError(f"{snapshot}: {overlap_count} claims occur in both splits.")

    audit_rows.append({
        "snapshot": snapshot,
        "rows": len(df),
        "columns_input": len(original_df.columns),
        "train_rows": int(train_mask.sum()),
        "test_rows": int(test_mask.sum()),
        "accident_year_min": int(df["ACC_YEAR"].min()),
        "accident_year_max": int(df["ACC_YEAR"].max()),
        "duplicate_claim_rows": duplicate_claims,
        "train_test_claim_overlap": overlap_count,
        "positive_latest_bixs_claims": int(df[TARGET_CLASS_CLEAN].sum()),
        "snapshot_bixs_total": float(df[SNAPSHOT_EXCESS].sum()),
        "latest_observed_bixs_total": float(df[TARGET_LATEST].sum()),
        "future_development_total": float(df[TARGET_FUTURE_CLEAN].sum()),
        "negative_future_development_claims": int(
            (df[TARGET_FUTURE_CLEAN] < 0).sum()
        ),
        "upstream_future_target_max_abs_diff": target_check_max_abs_diff,
        "upstream_future_target_mismatch_count": target_check_mismatch_count,
        "upstream_latest_flag_mismatch_count": class_flag_mismatch_count,
    })

    cleaned_data[snapshot] = df

audit_df = pd.DataFrame(audit_rows)
display(audit_df)

## Save predictors, targets and metadata separately

This is the main leakage safeguard. The modelling notebook will never receive
outcome or split columns inside `X_train` or `X_test`.



This is the stage where the checked snapshot data are organised into clean training and testing sets ready for the models.

In [ ]:
for snapshot, df in cleaned_data.items():
    snapshot_folder = OUTPUT_FOLDER / snapshot
    snapshot_folder.mkdir(parents=True, exist_ok=True)

    train_df = df.loc[df["valuation_split"].eq(TRAIN_LABEL)].copy()
    test_df = df.loc[df["valuation_split"].eq(TEST_LABEL)].copy()

    meta_columns = ["MODEL_ROW_ID"] + [
        col for col in META_CANDIDATES if col in df.columns
    ]

    X_train = train_df[FEATURE_COLUMNS].reset_index(drop=True)
    X_test = test_df[FEATURE_COLUMNS].reset_index(drop=True)

    targets_train = train_df[
        [SNAPSHOT_EXCESS, TARGET_LATEST, TARGET_FUTURE_CLEAN, TARGET_CLASS_CLEAN]
    ].reset_index(drop=True)
    targets_test = test_df[
        [SNAPSHOT_EXCESS, TARGET_LATEST, TARGET_FUTURE_CLEAN, TARGET_CLASS_CLEAN]
    ].reset_index(drop=True)

    meta_train = train_df[meta_columns].reset_index(drop=True)
    meta_test = test_df[meta_columns].reset_index(drop=True)

    assert len(X_train) == len(targets_train) == len(meta_train)
    assert len(X_test) == len(targets_test) == len(meta_test)
    assert list(X_train.columns) == FEATURE_COLUMNS
    assert list(X_test.columns) == FEATURE_COLUMNS

    X_train.to_parquet(snapshot_folder / "X_train.parquet", index=False)
    X_test.to_parquet(snapshot_folder / "X_test.parquet", index=False)
    targets_train.to_parquet(
        snapshot_folder / "targets_train.parquet", index=False
    )
    targets_test.to_parquet(
        snapshot_folder / "targets_test.parquet", index=False
    )
    meta_train.to_parquet(snapshot_folder / "meta_train.parquet", index=False)
    meta_test.to_parquet(snapshot_folder / "meta_test.parquet", index=False)

    print(
        snapshot,
        "saved:",
        f"X_train={X_train.shape}",
        f"X_test={X_test.shape}"
    )

This part  creates a permanent record of what went into the models and the checks performed, so the modelling setup can be reviewed and reproduced consistently later.

In [ ]:
feature_manifest = {
    "snapshots": SNAPSHOTS,
    "train_label": TRAIN_LABEL,
    "test_label": TEST_LABEL,
    "categorical_features": CATEGORICAL_FEATURES,
    "numeric_features": NUMERIC_FEATURES,
    "feature_columns": FEATURE_COLUMNS,
    "target_latest": TARGET_LATEST,
    "target_future": TARGET_FUTURE_CLEAN,
    "target_classification": TARGET_CLASS_CLEAN,
    "snapshot_excess": SNAPSHOT_EXCESS,
    "forbidden_features": sorted(FORBIDDEN_FEATURES),
    "notes": [
        "Predictors are defined by whitelist.",
        "Targets and metadata are saved separately from X.",
        "DEV_QTR is excluded because separate fixed-snapshot models are fitted.",
        "Raw and LOG1P monetary copies are not duplicated.",
        "CLASS, cover, NATLOSS and ACC_QTR are categorical.",
    ],
}

with open(OUTPUT_FOLDER / "feature_manifest.json", "w") as f:
    json.dump(feature_manifest, f, indent=2)

audit_df.to_csv(
    OUTPUT_FOLDER / "clean_snapshot_input_audit.csv",
    index=False
)

print("Saved manifest and audit to:", OUTPUT_FOLDER)